# План ноутбука

## Цель  
Пошагово отладить пред-обработку одного КТ-исследования (DICOM-серия и NIfTI-файл).

## Шаги  

1. **Загрузка данных**  
   – загрузка 3D объёма из DICOM-папки и из `*.nii.gz`  
   – вывод формы, dtype, affine (для NIfTI)

2. **Приведение к HU + нормализация**  
   – окно HU [–1,000;400] → масштабирование в [0 ; 1]

3. **Унификация voxel-spacing**  
   – ресэмплинг к isotropic spacing 1.5 mm

4. **Кроп лёгких**  
   – быстрая mask-based bbox (Threshold HU<–300 → morphology)

5. **Шейпинг**  
   – `Resize/Pad` до фиксированного куба 128×128×128

6. **Tensor + сохранение**  
   – сохранить результат `SaveImage(writer="TorchTensor")`

7. **Визуализация чек-пойнтов**  
   – срезы, гистограммы интенсивности

## Методика  
-  один `monai.transforms.Compose`; после каждой трансформы — стоп-ячейка → смотрим вывод  
-  отладка на DICOM и NIfTI параллельно; затем масштабируем на весь датасет


# Шаг 0. Установка и импорт библиотек

In [ ]:
%pip install monai[all] itk-core pydicom nibabel SimpleITK

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "/kernel/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 105, in _run_wrapper
    status = _inner_run()
  File "/kernel/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 96, in _inner_run
    return self.run(options, args)
  File "/kernel/lib/python3.10/site-packages/pip/_internal/cli/req_command.py", line 68, in wrapper
    return func(self, options, args)
  File "/kernel/lib/python3.10/site-packages/pip/_internal/commands/install.py", line 387, in run
    requirement_set = resolver.resolve(
  File "/kernel/lib/python3.10/site-packages/pip/_internal/resolution/resolvelib/resolver.py", line 96, in resolve
    result = self._result = resolver.r

In [2]:
%pip uninstall -y itk itk-core itk-gdcm pydicom

Found existing installation: itk-core 5.4.4.post1
Uninstalling itk-core-5.4.4.post1:
  Successfully uninstalled itk-core-5.4.4.post1
Found existing installation: pydicom 3.0.1
Uninstalling pydicom-3.0.1:
  Successfully uninstalled pydicom-3.0.1


In [3]:
%pip install -qU itk pydicom


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# установим рабочую директорию
work_dir = os.path.join(os.path.expanduser('~'), 'work', 'data')

%cd {work_dir}

/home/jupyter/work/data


In [ ]:
directory_path =  Path("dataset_subset/pathology/dicom_studies/RLADD02000233198_RLSDD02000236024/RLADD02000233198_RLSDD02000236024") 

# Получаем список только файлов, используя .iterdir() и .is_file()
file_list = [entry for entry in directory_path.iterdir()]

for i in file_list: print(i)

# Модуль предобработки данных

In [1]:
import os
import gc
import json
import hashlib
import warnings
import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Union
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
import contextlib

import numpy as np
import torch
import SimpleITK as sitk
import pydicom
import nibabel as nib
from tqdm import tqdm

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    ScaleIntensityRanged, Lambda
)
from monai.data import Dataset, DataLoader

# Подавление warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# Логирование
# ----------------------------
class CTLogger:
    def __init__(self, name: str = "CTPreprocessor", level: str = "INFO", 
                 enable_console: bool = True, log_file: Optional[str] = None):
        self.logger = logging.getLogger(name)
        self.logger.setLevel(getattr(logging, level.upper()))
        
        for handler in self.logger.handlers[:]:
            self.logger.removeHandler(handler)
        
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        
        if enable_console:
            console_handler = logging.StreamHandler()
            console_handler.setFormatter(formatter)
            self.logger.addHandler(console_handler)
        
        if log_file:
            file_handler = logging.FileHandler(log_file)
            file_handler.setFormatter(formatter)
            self.logger.addHandler(file_handler)
    
    def debug(self, msg): self.logger.debug(msg)
    def info(self, msg): self.logger.info(msg)
    def warning(self, msg): self.logger.warning(msg)
    def error(self, msg): self.logger.error(msg)

# ----------------------------
# JSON сериализация (ИСПРАВЛЕНО)
# ----------------------------
def make_json_serializable(obj: Any) -> Any:
    """Исправленная JSON сериализация"""
    if isinstance(obj, (str, int, float, bool, type(None))):
        return obj
    elif isinstance(obj, Path):
        return str(obj)  # ИСПРАВЛЕНО: PosixPath -> str
    elif isinstance(obj, np.ndarray):
        return obj.tolist() if obj.size < 100 else f"<array shape={obj.shape}>"
    elif isinstance(obj, torch.Tensor):
        return f"<tensor shape={tuple(obj.shape)}>"
    elif isinstance(obj, (list, tuple)):
        return [make_json_serializable(item) for item in obj]
    elif isinstance(obj, dict):
        return {key: make_json_serializable(value) for key, value in obj.items()}
    else:
        return str(obj)

def safe_json_dump(data: Dict[str, Any], filepath: Path) -> None:
    """Безопасное сохранение JSON"""
    serializable_data = make_json_serializable(data)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(serializable_data, f, indent=2, ensure_ascii=False)

# ----------------------------
# Конфигурация (совместимая с вашим API)
# ----------------------------
@dataclass
class PreprocConfig:
    input_root: str
    preproc_root: str
    
    # Геометрия
    target_pixdim: Tuple[float, float, float] = (0.8, 0.8, 0.8)
    target_size: Tuple[int, int, int] = (128, 128, 128)
    
    # HU параметры
    hu_window: Tuple[float, float] = (-1000.0, 400.0)
    padding_value: Optional[float] = -2048.0
    
    # QA (ЛИБЕРАЛЬНЫЕ пороги)
    enable_qa: bool = True
    min_depth: int = 16          # Снижено с 64
    max_clip_share: float = 0.95  # Увеличено с 0.25
    max_padding_share: float = 0.90  # Увеличено с 0.50
    
    # DICOM фильтры (ОТКЛЮЧЕНЫ по умолчанию)
    enable_dicom_filters: bool = False
    reject_localizer: bool = False
    require_ct_modality: bool = False
    
    # Производительность
    n_procs: int = 4
    itk_threads: int = 1
    overwrite: bool = False
    
    # Логирование
    pipeline_version: str = "v3.0"
    enable_logging: bool = True
    log_level: str = "INFO"
    log_file: Optional[str] = None

def compute_pipeline_hash(config: PreprocConfig) -> str:
    key_params = {
        "version": config.pipeline_version,
        "pixdim": config.target_pixdim,
        "size": config.target_size,
        "hu_window": config.hu_window,
        "padding": config.padding_value,
    }
    content = json.dumps(key_params, sort_keys=True)
    return hashlib.md5(content.encode()).hexdigest()[:12]

# ----------------------------
# Проверенные функции загрузки (ВАШИ)
# ----------------------------
def select_max_depth_uid(dicom_dir: str) -> Tuple[str, List[str]]:
    """Выбор серии с максимальным количеством срезов"""
    try:
        reader = sitk.ImageSeriesReader()
        series_uids = reader.GetGDCMSeriesIDs(str(dicom_dir))
        
        max_uid = ""
        max_files = []
        
        for uid in series_uids:
            filenames = reader.GetGDCMSeriesFileNames(str(dicom_dir), uid)
            if len(filenames) > len(max_files):
                max_uid = uid
                max_files = filenames
                
        return max_uid, max_files
        
    except Exception:
        # Fallback: все .dcm файлы
        dcm_files = list(Path(dicom_dir).glob("*.dcm"))
        return "fallback_series", [str(f) for f in dcm_files]

def load_via_filelist_sitk(filelist: List[str]) -> Tuple[torch.Tensor, Dict[str, Any]]:
    """Проверенная загрузка DICOM с корректными метаданными"""
    r = sitk.ImageSeriesReader()
    r.SetFileNames(filelist)
    img = r.Execute()
    arr = sitk.GetArrayFromImage(img)  # (D,H,W)
    arr = np.moveaxis(arr, 0, -1)[None, None]  # (1,1,H,W,D)
    vol = torch.from_numpy(arr.copy().astype(np.float32))
    
    spacing = img.GetSpacing()
    direction = img.GetDirection()
    origin = img.GetOrigin()
    
    # Создаем affine матрицу
    affine = np.eye(4, dtype=np.float64)
    if len(direction) == 9:
        direction_matrix = np.array(direction).reshape(3, 3)
        spacing_matrix = np.diag(spacing)
        affine[:3, :3] = direction_matrix @ spacing_matrix
        affine[:3, 3] = origin
    
    meta = {
        "spacing": spacing,
        "direction": direction,
        "origin": origin,
        "affine": affine,
        "spatial_shape": vol.shape[2:],
        "reader": "SimpleITKSeriesReader"
    }
    return vol, meta

def robust_load_dicom_volume(dicom_dir: str, logger: CTLogger) -> Tuple[torch.Tensor, Dict[str, Any]]:
    """Робастная загрузка DICOM с fallback"""
    try:
        logger.debug(f"📖 Загрузка DICOM: {Path(dicom_dir).name}")
        
        uid, filelist = select_max_depth_uid(dicom_dir)
        logger.debug(f"  🔍 Серия: {uid[:50]}..., файлов: {len(filelist)}")
        
        # Прямая загрузка через SimpleITK (надежнее чем ITKReader)
        vol, meta = load_via_filelist_sitk(filelist)
        logger.debug(f"  ✅ Загружено: {vol.shape}")
        
        return vol, meta
        
    except Exception as e:
        logger.error(f"❌ Ошибка загрузки DICOM {dicom_dir}: {e}")
        raise RuntimeError(f"Не удалось загрузить DICOM: {e}")

def load_nifti_robust(filepath: str, logger: CTLogger) -> Tuple[torch.Tensor, Dict[str, Any]]:
    """Робастная загрузка NIfTI с fallback на nibabel"""
    try:
        logger.debug(f"📖 Загрузка NIfTI: {Path(filepath).name}")
        
        # Попытка через MONAI
        try:
            loader = Compose([
                LoadImaged(keys=["image"], image_only=False),
                EnsureChannelFirstd(keys=["image"]),
                EnsureTyped(keys=["image"], dtype=torch.float32),
            ])
            
            data = {"image": filepath}
            result = loader(data)
            volume = result["image"]
            
            meta = {}
            if hasattr(volume, 'meta'):
                meta = dict(volume.meta)
            
            # Нормализация размерности
            if len(volume.shape) == 3:  # (H,W,D)
                volume = volume.unsqueeze(0).unsqueeze(0)
            elif len(volume.shape) == 4:  # (C,H,W,D)
                volume = volume.unsqueeze(0)
            
            logger.debug(f"  ✅ MONAI загрузка: {volume.shape}")
            return volume, meta
            
        except Exception as monai_error:
            logger.debug(f"  🔄 MONAI ошибка: {monai_error}, пробуем nibabel")
            
            # Fallback через nibabel
            nii = nib.load(filepath)
            array = nii.get_fdata().astype(np.float32)
            
            if len(array.shape) == 3:  # (H,W,D)
                array = array[None, None, ...]  # (1,1,H,W,D)
            
            volume = torch.from_numpy(array)
            meta = {
                "affine": nii.affine,
                "spacing": [1.0, 1.0, 1.0],  # default
                "reader": "nibabel_fallback"
            }
            
            logger.debug(f"  ✅ Nibabel загрузка: {volume.shape}")
            return volume, meta
        
    except Exception as e:
        logger.error(f"❌ Полная ошибка загрузки NIfTI {filepath}: {e}")
        raise RuntimeError(f"Не удалось загрузить NIfTI: {e}")

# ----------------------------
# Обнаружение данных (ИСПРАВЛЕНО)
# ----------------------------
def discover_inputs_robust(input_root: Union[str, Path], logger: CTLogger) -> List[Dict[str, Any]]:
    """ИСПРАВЛЕННОЕ обнаружение входных данных"""
    input_root = Path(input_root)  # ИСПРАВЛЕНО: убрал [0]
    inputs = []
    
    logger.info(f"🔍 Сканирование: {input_root}")
    
    if not input_root.exists():
        logger.error(f"❌ Директория не существует: {input_root}")
        return inputs
    
    # NIfTI файлы
    logger.info("📁 Поиск NIfTI...")
    nifti_count = 0
    for pattern in ["*.nii", "*.nii.gz", "*.NII", "*.NII.GZ"]:
        for nii_file in input_root.rglob(pattern):
            if nii_file.is_file() and nii_file.stat().st_size > 1024:
                inputs.append({
                    "kind": "nifti",
                    "path": str(nii_file),
                    "source_id": f"nifti::{nii_file.name}",
                })
                nifti_count += 1
    
    logger.info(f"  ✅ NIfTI: {nifti_count}")
    
    # DICOM директории
    logger.info("📁 Поиск DICOM...")
    dicom_dirs = set()
    for pattern in ["*.dcm", "*.DCM"]:
        for dcm_file in input_root.rglob(pattern):
            if dcm_file.is_file() and dcm_file.stat().st_size > 512:
                dicom_dirs.add(dcm_file.parent)
    
    dicom_count = 0
    for dcm_dir in dicom_dirs:
        try:
            uid, filelist = select_max_depth_uid(str(dcm_dir))
            
            if len(filelist) >= 5:  # Минимум 5 файлов
                inputs.append({
                    "kind": "dicom",
                    "root": str(dcm_dir),
                    "series_uid": uid,
                    "filelist": filelist,
                    "source_id": f"dicom::{dcm_dir.name}::{uid[:12]}",
                })
                dicom_count += 1
            
        except Exception as e:
            logger.debug(f"  ⚠️ Пропуск {dcm_dir}: {e}")
            continue
    
    logger.info(f"  ✅ DICOM: {dicom_count}")
    logger.info(f"🎯 Всего: {len(inputs)}")
    
    return inputs

# ----------------------------
# Робастный пайплайн (БЕЗ проблемных MONAI трансформов)
# ----------------------------
def build_robust_pipeline(config: PreprocConfig) -> Compose:
    """Робастный пайплайн без проблемных MONAI трансформов"""
    
    def robust_preprocessing(data):
        img = data["image"]
        meta = data.get("image_meta_dict", {})
        
        print(f"  🔍 Вход: {img.shape}, мета: {len(meta)}")
        
        # 1. Padding замена
        if config.padding_value is not None:
            padding_mask = (img == config.padding_value)
            if padding_mask.sum() > 0:
                img = torch.where(padding_mask, 
                                torch.tensor(config.hu_window[0], dtype=img.dtype), 
                                img)
                print(f"  🔄 Заменено padding: {padding_mask.sum().item()} вокселей")
        
        # 2. HU клиппинг
        img_clipped = torch.clamp(img, config.hu_window[0], config.hu_window[1])
        
        # 3. Spacing нормализация (если возможно)
        img_resampled = img_clipped
        try:
            current_spacing = meta.get("spacing", None)
            
            if current_spacing is not None:
                spacing_array = np.array(current_spacing)
                target_array = np.array(config.target_pixdim)
                spacing_diff = np.abs(spacing_array - target_array).max()
                
                if spacing_diff > 0.1:  # Нужен ресэмплинг
                    print(f"  📏 Ресэмплинг: {current_spacing} -> {config.target_pixdim}")
                    
                    # Извлекаем numpy массив
                    if len(img_clipped.shape) == 5:
                        numpy_array = img_clipped[0, 0].detach().cpu().numpy()
                    elif len(img_clipped.shape) == 4:
                        numpy_array = img_clipped[0].detach().cpu().numpy()
                    else:
                        numpy_array = img_clipped.detach().cpu().numpy()
                    
                    # SimpleITK ресэмплинг
                    sitk_img = sitk.GetImageFromArray(numpy_array.transpose(2, 1, 0))
                    sitk_img.SetSpacing(current_spacing)
                    
                    if "origin" in meta:
                        sitk_img.SetOrigin(meta["origin"])
                    if "direction" in meta:
                        sitk_img.SetDirection(meta["direction"])
                    
                    resampler = sitk.ResampleImageFilter()
                    resampler.SetOutputSpacing(config.target_pixdim)
                    resampler.SetInterpolator(sitk.sitkLinear)
                    
                    original_size = sitk_img.GetSize()
                    new_size = [
                        max(1, int(original_size[i] * current_spacing[i] / config.target_pixdim[i]))
                        for i in range(3)
                    ]
                    resampler.SetSize(new_size)
                    resampler.SetOutputOrigin(sitk_img.GetOrigin())
                    resampler.SetOutputDirection(sitk_img.GetDirection())
                    
                    resampled_sitk = resampler.Execute(sitk_img)
                    resampled_array = sitk.GetArrayFromImage(resampled_sitk).transpose(2, 1, 0)
                    resampled_tensor = torch.from_numpy(resampled_array.astype(np.float32))
                    
                    # Восстановление размерностей
                    if len(img_clipped.shape) == 5:
                        resampled_tensor = resampled_tensor.unsqueeze(0).unsqueeze(0)
                    elif len(img_clipped.shape) == 4:
                        resampled_tensor = resampled_tensor.unsqueeze(0)
                    
                    img_resampled = resampled_tensor
                    print(f"  ✅ Ресэмплинг выполнен: {img_resampled.shape}")
                else:
                    print(f"  ✅ Spacing подходящий")
            else:
                print(f"  ⚠️ Нет данных spacing")
                
        except Exception as e:
            print(f"  ❌ Ошибка ресэмплинга: {e}")
            img_resampled = img_clipped
        
        # 4. Интенсивностная нормализация
        img_normalized = (img_resampled - config.hu_window[0]) / (config.hu_window[1] - config.hu_window[0])
        
        # 5. Resize через F.interpolate (БЕЗОПАСНО)
        import torch.nn.functional as F
        
        current_size = img_normalized.shape[-3:]
        if current_size != config.target_size:
            resized = F.interpolate(
                img_normalized, 
                size=config.target_size, 
                mode='trilinear', 
                align_corners=False
            )
            print(f"  📐 Resize: {current_size} -> {config.target_size}")
        else:
            resized = img_normalized
            print(f"  ✅ Размер подходящий")
        
        data["image"] = resized
        return data
    
    return Compose([
        Lambda(func=robust_preprocessing),
        EnsureTyped(keys=["image"], dtype=torch.float32),
    ])

# ----------------------------
# QA проверки (ЛИБЕРАЛЬНЫЕ)
# ----------------------------
def qa_volume_liberal(volume: torch.Tensor, config: PreprocConfig) -> Dict[str, Any]:
    """Либеральная QA проверка"""
    if not config.enable_qa:
        return {"qa_pass": True, "reason": "qa_disabled"}
    
    # Быстрые проверки
    shape = volume.shape
    depth = shape[-1] if len(shape) >= 3 else 1
    
    depth_ok = depth >= config.min_depth
    
    # Сэмплирование для скорости
    sample_vol = volume.view(-1)[::1000]
    
    # Анализ на выборке
    clipped = ((sample_vol <= 0.01) | (sample_vol >= 0.99)).float().mean()
    air_ratio = (sample_vol <= 0.1).float().mean()
    
    clip_ok = clipped <= config.max_clip_share
    air_ok = air_ratio <= config.max_padding_share
    
    qa_pass = depth_ok and clip_ok and air_ok
    
    return {
        "qa_pass": qa_pass,
        "depth_ok": depth_ok,
        "clip_ok": clip_ok,
        "air_ok": air_ok,
        "metrics": {
            "depth": depth,
            "clip_share": clipped.item(),
            "air_share": air_ratio.item(),
        }
    }

# ----------------------------
# Главный процессор (ФИНАЛЬНЫЙ)
# ----------------------------
def process_one_sample_final(sample: Dict[str, Any], config: PreprocConfig, 
                           pipeline_hash: str, logger: CTLogger) -> Dict[str, Any]:
    """Финальная обработка одного образца"""
    
    sample_id = sample.get("source_id", "unknown")
    
    try:
        logger.debug(f"🔄 Обработка: {sample_id}")
        
        # Проверка кэша
        output_dir = Path(config.preproc_root) / pipeline_hash
        file_hash = hashlib.md5(sample_id.encode()).hexdigest()[:8]
        output_file = output_dir / f"{file_hash}.nii.gz"
        manifest_file = output_dir / f"{file_hash}.json"
        
        if output_file.exists() and manifest_file.exists() and not config.overwrite:
            logger.debug(f"  💾 Кэш: {output_file.name}")
            with open(manifest_file, 'r') as f:
                cached = json.load(f)
            return {"ok": True, "cached": True, **cached}
        
        # Загрузка
        if sample["kind"] == "nifti":
            volume, meta = load_nifti_robust(sample["path"], logger)
        else:
            volume, meta = robust_load_dicom_volume(sample["root"], logger)
        
        # Препроцессинг
        pipeline = build_robust_pipeline(config)
        data_dict = {"image": volume, "image_meta_dict": meta}
        
        # Очистка памяти
        del volume
        gc.collect()
        
        processed_data = pipeline(data_dict)
        processed_volume = processed_data["image"]
        
        logger.debug(f"  📊 Результат: {processed_volume.shape}")
        
        # QA
        qa_result = qa_volume_liberal(processed_volume, config)
        if not qa_result["qa_pass"]:
            logger.debug(f"  ❌ QA отклонен")
            return {
                "ok": False, "cached": False,
                "reason": "qa_failed",
                "source_id": sample_id,
                "qa": qa_result
            }
        
        # Сохранение
        output_dir.mkdir(parents=True, exist_ok=True)
        
        if len(processed_volume.shape) == 5:
            save_array = processed_volume[0, 0].detach().cpu().numpy()
        elif len(processed_volume.shape) == 4:
            save_array = processed_volume[0].detach().cpu().numpy()
        else:
            save_array = processed_volume.detach().cpu().numpy()
        
        affine = np.eye(4)
        if "affine" in meta:
            try:
                affine = np.array(meta["affine"])
            except:
                pass
        
        nii_img = nib.Nifti1Image(save_array.astype(np.float32), affine)
        nib.save(nii_img, str(output_file))
        
        # Манифест
        manifest = {
            "source_id": sample_id,
            "kind": sample["kind"],
            "output_img": str(output_file),
            "pipe_hash": pipeline_hash,
            "qa": qa_result,
        }
        
        safe_json_dump(manifest, manifest_file)
        
        # Очистка
        del processed_volume
        gc.collect()
        
        logger.debug(f"  ✅ Сохранено: {output_file.name}")
        
        return {
            "ok": True, "cached": False,
            "source_id": sample_id,
            "output_img": str(output_file),
            "qa": qa_result
        }
        
    except Exception as e:
        logger.error(f"  ❌ Ошибка {sample_id}: {e}")
        return {
            "ok": False, "cached": False,
            "reason": f"exception: {str(e)}",
            "source_id": sample_id
        }

class MedPreprocessor:
    """ФИНАЛЬНАЯ версия процессора"""
    
    def __init__(self, cfg: PreprocConfig):
        self.cfg = cfg
        self.pipe_hash = compute_pipeline_hash(cfg)
        
        self.logger = CTLogger(
            name="MedPreprocessor",
            level=cfg.log_level if cfg.enable_logging else "ERROR",
            enable_console=cfg.enable_logging,
            log_file=cfg.log_file
        )
        
        self.output_dir = Path(cfg.preproc_root) / self.pipe_hash
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.logger.info(f"🚀 MedPreprocessor v3.0 инициализирован")
        self.logger.info(f"   📂 {cfg.input_root}")
        self.logger.info(f"   📁 {self.output_dir}")
    
    def build(self) -> List[Dict[str, Any]]:
        """Главная функция препроцессинга"""
        
        self.logger.info("🔍 Обнаружение данных...")
        samples = discover_inputs_robust(self.cfg.input_root, self.logger)
        
        if not samples:
            self.logger.warning("⚠️ Данные не найдены!")
            return []
        
        self.logger.info(f"📊 К обработке: {len(samples)}")
        
        results = []
        
        if self.cfg.n_procs <= 1:
            # Однопоточно
            for sample in tqdm(samples, desc="Обработка", disable=not self.cfg.enable_logging):
                result = process_one_sample_final(sample, self.cfg, self.pipe_hash, self.logger)
                results.append(result)
        else:
            # Многопоточно  
            process_func = partial(
                process_one_sample_final,
                config=self.cfg,
                pipeline_hash=self.pipe_hash,
                logger=self.logger
            )
            
            with ProcessPoolExecutor(max_workers=self.cfg.n_procs) as executor:
                futures = [executor.submit(process_func, sample) for sample in samples]
                
                for future in tqdm(as_completed(futures), total=len(samples), 
                                 desc="Обработка", disable=not self.cfg.enable_logging):
                    try:
                        result = future.result(timeout=300)
                        results.append(result)
                    except Exception as e:
                        self.logger.error(f"❌ Процесс: {e}")
                        results.append({"ok": False, "reason": f"process_error: {str(e)}"})
        
        # Статистика
        successful = sum(1 for r in results if r.get("ok", False))
        cached = sum(1 for r in results if r.get("cached", False))
        
        self.logger.info(f"✅ Завершено:")
        self.logger.info(f"   📊 Успешно: {successful}/{len(results)}")
        self.logger.info(f"   💾 Кэш: {cached}")
        
        # Индекс
        index_data = {
            "total": len(samples),
            "successful": successful,
            "cached": cached,
            "failed": len(results) - successful,
            "items": results
        }
        
        index_file = self.output_dir / "_index.json"
        safe_json_dump(index_data, index_file)
        
        return results
    
    def get_dataloader(self, batch_size: int = 4, num_workers: int = 2, 
                      shuffle: bool = False) -> DataLoader:
        """DataLoader для обучения"""
        
        index_file = self.output_dir / "_index.json"
        if not index_file.exists():
            raise FileNotFoundError(f"Индекс не найден: {index_file}")
        
        with open(index_file, 'r') as f:
            index_data = json.load(f)
        
        successful_items = [
            item for item in index_data["items"] 
            if item.get("ok", False) and "output_img" in item
        ]
        
        if not successful_items:
            raise ValueError("Нет данных для DataLoader")
        
        self.logger.info(f"📚 DataLoader: {len(successful_items)} файлов")
        
        data_dicts = [{"image": item["output_img"]} for item in successful_items]
        
        loader_transforms = Compose([
            LoadImaged(keys=["image"], image_only=True),
            EnsureChannelFirstd(keys=["image"]),
            EnsureTyped(keys=["image"], dtype=torch.float32),
        ])
        
        dataset = Dataset(data_dicts, transform=loader_transforms)
        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=min(num_workers, len(successful_items)),
            pin_memory=torch.cuda.is_available(),
            persistent_workers=(num_workers > 0),
        )
        
        return dataloader


Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
2025-09-27 16:51:48.149123: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-27 16:51:48.209315: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the

In [2]:
# ================================
# ЯЧЕЙКА 1: ИМПОРТ ФИНАЛЬНОГО МОДУЛЯ
# ================================
import sys
from pathlib import Path
import torch
import gc

# Импорт финального модуля
# (сохраните код выше как ct_preprocessor_final.py)
#from ct_preprocessor_final import PreprocConfig, MedPreprocessor

print("✅ Финальный модуль импортирован")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")


✅ Финальный модуль импортирован
PyTorch: 2.8.0+cu128
CUDA: False


In [3]:
# ================================
# ЯЧЕЙКА 2: ФИНАЛЬНАЯ КОНФИГУРАЦИЯ
# ================================

# УСТАНОВИТЕ СВОЙ ПУТЬ
YOUR_DATA_PATH = '/home/jupyter/work/data/dataset_subset/test_subset_small'  # ИЗМЕНИТЕ НА СВОЙ ПУТЬ
OUTPUT_PATH = '/home/jupyter/work/data/preproc'

# Финальная проверенная конфигурация
final_config = PreprocConfig(
    input_root=YOUR_DATA_PATH,
    preproc_root=OUTPUT_PATH,
    
    # Умеренные параметры для надежности
    target_pixdim=(1.0, 1.0, 1.5),    # Не слишком агрессивный ресэмплинг
    target_size=(96, 96, 64),         # Разумный размер
    hu_window=(-1000.0, 400.0),       # Стандартное легочное окно
    padding_value=-2048.0,            # Обработка padding
    
    # ЛИБЕРАЛЬНЫЕ QA настройки (основаны на опыте)
    enable_qa=True,
    min_depth=16,                     # Минимум 16 срезов
    max_clip_share=0.95,             # 95% клиппинга OK
    max_padding_share=0.90,          # 90% воздуха OK
    
    # Отключенные фильтры (для максимальной совместимости)
    enable_dicom_filters=False,
    reject_localizer=False,
    require_ct_modality=False,
    
    # Производительность
    n_procs=4,                       # Консервативно для стабильности
    overwrite=True,
    
    # Подробное логирование
    enable_logging=True,
    log_level="INFO",
    pipeline_version="v3.0_final",
)

print("🔧 Финальная конфигурация готова")
print(f"   📂 Вход: {final_config.input_root}")
print(f"   📁 Выход: {final_config.preproc_root}")
print(f"   📏 Размер: {final_config.target_size}")
print(f"   🔍 QA: {'включена' if final_config.enable_qa else 'отключена'}")

🔧 Финальная конфигурация готова
   📂 Вход: /home/jupyter/work/data/dataset_subset/test_subset_small
   📁 Выход: /home/jupyter/work/data/preproc
   📏 Размер: (96, 96, 64)
   🔍 QA: включена


In [4]:
# ================================
# ЯЧЕЙКА 3: ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ
# ================================

print("🚀 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ")
print("="*50)

# Создание процессора
final_processor = MedPreprocessor(final_config)

# Запуск полного препроцессинга
print("🔄 Запуск препроцессинга...")
final_results = final_processor.build()

# Детальный анализ результатов
successful = sum(1 for r in final_results if r.get("ok", False))
cached = sum(1 for r in final_results if r.get("cached", False))
failed = len(final_results) - successful

print(f"\n📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
print(f"   ✅ Успешно: {successful}")
print(f"   💾 Из кэша: {cached}")
print(f"   ❌ Ошибок: {failed}")
print(f"   📈 Успешность: {successful/len(final_results)*100 if final_results else 0:.1f}%")

# Анализ типов ошибок
if failed > 0:
    print(f"\n🔍 АНАЛИЗ ОШИБОК:")
    error_types = {}
    for result in final_results:
        if not result.get("ok", False):
            reason = result.get("reason", "unknown")
            error_types[reason] = error_types.get(reason, 0) + 1
    
    for reason, count in error_types.items():
        print(f"   {reason}: {count}")

# Тестирование DataLoader
if successful > 0:
    print(f"\n🎯 ТЕСТИРОВАНИЕ DATALOADER:")
    try:
        train_loader = final_processor.get_dataloader(
            batch_size=2,
            num_workers=4,
            shuffle=True
        )
        
        print(f"   ✅ DataLoader создан: {len(train_loader)} батчей")
        
        # Тест первого батча
        for batch in train_loader:
            image = batch["image"]
            print(f"   📊 Батч: {image.shape}")
            print(f"   🔢 Тип: {image.dtype}")
            print(f"   📏 Диапазон: [{image.min():.3f}, {image.max():.3f}]")
            
            # Проверки
            assert image.shape[0] <= 2, "Размер батча"
            assert len(image.shape) == 5, "Размерность"
            assert 0.0 <= image.min() and image.max() <= 1.0, "Диапазон значений"
            
            print(f"   ✅ Все проверки пройдены!")
            break
            
    except Exception as e:
        print(f"   ❌ Ошибка DataLoader: {e}")

print(f"\n🎉 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ ЗАВЕРШЕНО!")

if successful > 0:
    print(f"✅ СИСТЕМА ГОТОВА К ОБУЧЕНИЮ АВТОЭНКОДЕРА!")
    print(f"   📁 Данные: {final_processor.output_dir}")
    print(f"   📊 Образцов: {successful}")
    print(f"   📏 Размер: {final_config.target_size}")
else:
    print(f"❌ ТРЕБУЕТСЯ ДОПОЛНИТЕЛЬНАЯ НАСТРОЙКА")
    print(f"   Попробуйте:")
    print(f"   - enable_qa=False")
    print(f"   - Другие пути к данным")
    print(f"   - Проверить формат файлов")


🚀 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ


2025-09-27 16:51:51,321 - MedPreprocessor - INFO - 🚀 MedPreprocessor v3.0 инициализирован
2025-09-27 16:51:51,322 - MedPreprocessor - INFO -    📂 /home/jupyter/work/data/dataset_subset/test_subset_small
2025-09-27 16:51:51,323 - MedPreprocessor - INFO -    📁 /home/jupyter/work/data/preproc/6b8cb4640fc9
2025-09-27 16:51:51,324 - MedPreprocessor - INFO - 🔍 Обнаружение данных...
2025-09-27 16:51:51,325 - MedPreprocessor - INFO - 🔍 Сканирование: /home/jupyter/work/data/dataset_subset/test_subset_small
2025-09-27 16:51:51,327 - MedPreprocessor - INFO - 📁 Поиск NIfTI...


🔄 Запуск препроцессинга...


2025-09-27 16:51:51,732 - MedPreprocessor - INFO -   ✅ NIfTI: 10
2025-09-27 16:51:51,733 - MedPreprocessor - INFO - 📁 Поиск DICOM...
2025-09-27 16:51:54,261 - MedPreprocessor - INFO -   ✅ DICOM: 6
2025-09-27 16:51:54,262 - MedPreprocessor - INFO - 🎯 Всего: 16
2025-09-27 16:51:54,263 - MedPreprocessor - INFO - 📊 К обработке: 16
Обработка:   0%|          | 0/16 [00:00<?, ?it/s]

  🔍 Вход: torch.Size([1, 1, 512, 512, 35]), мета: 43
  🔍 Вход: torch.Size([1, 1, 512, 512, 39]), мета: 43
  🔍 Вход: torch.Size([1, 1, 512, 512, 40]), мета: 43
  🔍 Вход: torch.Size([1, 1, 512, 512, 43]), мета: 43
  🔄 Заменено padding: 1968820 вокселей
  ⚠️ Нет данных spacing  🔄 Заменено padding: 2250080 вокселей

  🔄 Заменено padding: 2193828 вокселей
  📐 Resize: torch.Size([512, 512, 35]) -> (96, 96, 64)
  ⚠️ Нет данных spacing
  ⚠️ Нет данных spacing
  🔄 Заменено padding: 2418836 вокселей
  📐 Resize: torch.Size([512, 512, 39]) -> (96, 96, 64)
  📐 Resize: torch.Size([512, 512, 40]) -> (96, 96, 64)
  ⚠️ Нет данных spacing
  📐 Resize: torch.Size([512, 512, 43]) -> (96, 96, 64)


...

  🔍 Вход: torch.Size([1, 1, 512, 512, 61]), мета: 6
  📏 Ресэмплинг: (0.607422, 0.607422, 5.0) -> (1.0, 1.0, 1.5)
  ✅ Ресэмплинг выполнен: torch.Size([1, 1, 311, 311, 203])
  📐 Resize: torch.Size([311, 311, 203]) -> (96, 96, 64)


Обработка:  88%|████████▊ | 14/16 [00:18<00:03,  1.92s/it]

  ✅ Ресэмплинг выполнен: torch.Size([1, 1, 400, 400, 240])
  📐 Resize: torch.Size([400, 400, 240]) -> (96, 96, 64)
  ✅ Ресэмплинг выполнен: torch.Size([1, 1, 351, 351, 203])
  📐 Resize: torch.Size([351, 351, 203]) -> (96, 96, 64)


Обработка: 100%|██████████| 16/16 [00:20<00:00,  1.31s/it]
2025-09-27 16:52:15,365 - MedPreprocessor - INFO - ✅ Завершено:
2025-09-27 16:52:15,366 - MedPreprocessor - INFO -    📊 Успешно: 16/16
2025-09-27 16:52:15,367 - MedPreprocessor - INFO -    💾 Кэш: 0
2025-09-27 16:52:15,390 - MedPreprocessor - INFO - 📚 DataLoader: 16 файлов



📊 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:
   ✅ Успешно: 16
   💾 Из кэша: 0
   ❌ Ошибок: 0
   📈 Успешность: 100.0%

🎯 ТЕСТИРОВАНИЕ DATALOADER:
   ✅ DataLoader создан: 8 батчей
   📊 Батч: torch.Size([2, 1, 96, 96, 64])
   🔢 Тип: torch.float32
   📏 Диапазон: [0.000, 1.000]
   ✅ Все проверки пройдены!

🎉 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ ЗАВЕРШЕНО!
✅ СИСТЕМА ГОТОВА К ОБУЧЕНИЮ АВТОЭНКОДЕРА!
   📁 Данные: /home/jupyter/work/data/preproc/6b8cb4640fc9
   📊 Образцов: 16
   📏 Размер: (96, 96, 64)
